# بناء مجموعة البيانات وتوليد صور المورف (Morph Generation) — DMorphNet

هذا الدفتر ينفّذ خطوة **"Dataset Construction and Morph Image Generation"** كاملة على Google Colab:

1. **تحميل الصور الحقيقية** من مجموعة `FaceMorph_EClub_Task` على Kaggle (وهي نفس صور CelebA — 202,599 صورة وجه حقيقية بإضاءات ووضعيات ومظاهر مختلفة).
2. **توليد 21,000 صورة مدموجة (Morphed)** بدمج ملامح وجهين لشخصين مختلفين في صورة واحدة — بنفس فكرة أداة AI FaceSwap (استخراج معالم الوجه، تقسيم Delaunay مثلثي، تشويه Affine ثم مزج الوجهين) بحيث تبدو النتيجة واقعية وليست مصطنعة.
3. **مراجعة وتصفية تلقائية** للصور المولّدة (التأكد من وجود وجه واضح وحدّة كافية) ثم **وضع الملصقات** (حقيقي / مدموج).
4. **بناء المجموعة النهائية: 45,000 صورة** = 24,000 حقيقية + 21,000 مدموجة.
5. **التقسيم**:

| القسم | الإجمالي | حقيقية | مدموجة |
|---|---|---|---|
| التدريب (Train) | 30,000 | 16,000 | 14,000 |
| التحقق (Validation) | 13,000 | 7,000 | 6,000 |
| المراقبة/الاختبار (Test) | 2,000 | 1,000 | 1,000 |

6. **ضمان عدالة التقييم**: لا تظهر نفس الهوية (نفس الشخص) في قسمين مختلفين أبداً — لا في الصور الحقيقية ولا كمصدر لصور المورف.

> ⚠️ ملاحظة: توليد 21,000 صورة مورف يستغرق وقتاً (١–٣ ساعات تقريباً على Colab). يمكنك أولاً تجربة أعداد صغيرة بتعديل خلية الإعدادات، ثم تشغيل الأعداد الكاملة.


## ١) تثبيت المكتبات المطلوبة

نثبّت:
- `kagglehub` لتحميل مجموعة البيانات من Kaggle مباشرة.
- `mediapipe` لاستخراج **468 معلماً (Landmark)** من كل وجه.
- `opencv` و `scipy` لعمليات التشويه (Warping) وتقسيم Delaunay.
- `tqdm` لعرض شريط التقدم أثناء التوليد.


In [ ]:
!pip -q install kagglehub mediapipe opencv-python-headless scipy tqdm pandas matplotlib
print("تم تثبيت جميع المكتبات بنجاح ✅")


## ٢) تحميل الصور الحقيقية من Kaggle

نحمّل مجموعة البيانات (202,599 صورة وجه حقيقية). المجموعة عامة على Kaggle لذا لا تحتاج مفتاح API.

> إذا كان لديك الرابط الدقيق لنسخة `FaceMorph_EClub_Task` على Kaggle، ضع اسمها في المتغيّر `DATASET_SLUG` (الصور نفسها هي صور CelebA المحاذاة).


In [ ]:
import kagglehub, os, glob

# اسم مجموعة البيانات على Kaggle (نفس صور FaceMorph_EClub_Task — 202,599 صورة)
DATASET_SLUG = "jessicali9530/celeba-dataset"

DATA_ROOT = kagglehub.dataset_download(DATASET_SLUG)
print("تم التحميل في المسار:", DATA_ROOT)

# البحث عن مجلد الصور
all_jpgs = glob.glob(os.path.join(DATA_ROOT, "**", "*.jpg"), recursive=True)
print("إجمالي عدد الصور الحقيقية:", len(all_jpgs))

IMG_DIR = os.path.dirname(all_jpgs[0])
print("مجلد الصور:", IMG_DIR)


## ٣) تحميل ملف الهويات (Identity File)

لضمان أن **نفس الشخص لا يظهر في التدريب والاختبار معاً**، نحتاج ملف `identity_CelebA.txt` الذي يربط كل صورة برقم هوية الشخص.

- الخلية تبحث عن الملف داخل ما تم تحميله تلقائياً.
- إذا لم تجده، سيُطلب منك **رفعه يدوياً** (متوفر مع النسخة الأصلية من CelebA).
- كحل أخير (غير مفضّل) يمكن اعتبار كل صورة هوية مستقلة، لكن حينها لا يكون ضمان فصل الهويات كاملاً.


In [ ]:
import pandas as pd

identity_path = None
for cand in glob.glob(os.path.join(DATA_ROOT, "**", "*identity*"), recursive=True):
    if os.path.isfile(cand):
        identity_path = cand
        break

if identity_path is None:
    print("لم يتم العثور على ملف الهويات — الرجاء رفع identity_CelebA.txt الآن:")
    from google.colab import files
    uploaded = files.upload()
    identity_path = list(uploaded.keys())[0]

print("ملف الهويات:", identity_path)

if identity_path.endswith(".txt"):
    ident = pd.read_csv(identity_path, sep=r"\s+", header=None,
                        names=["image_id", "identity"])
else:
    ident = pd.read_csv(identity_path)
    ident.columns = ["image_id", "identity"]

print("عدد الصور في ملف الهويات:", len(ident))
print("عدد الهويات (الأشخاص) المختلفة:", ident["identity"].nunique())
ident.head()


## ٤) الإعدادات العامة

هنا نحدد جميع الأرقام المطلوبة في الورقة:
- 24,000 صورة حقيقية + 21,000 صورة مدموجة = **45,000 صورة**.
- التقسيم: تدريب (16k حقيقي + 14k مورف)، تحقق (7k + 6k)، مراقبة/اختبار (1k + 1k).

> 💡 **للتجربة السريعة**: قبل التشغيل الكامل، صغّر الأرقام مؤقتاً (مثلاً 100 حقيقية و50 مورف لكل قسم) للتأكد أن كل شيء يعمل، ثم أعد الأرقام الأصلية.


In [ ]:
import random, shutil
import numpy as np

SEED = 42
random.seed(SEED)
np.random.seed(SEED)

# أعداد الصور المطلوبة لكل قسم (حسب الورقة العلمية)
SPLITS = {
    "train": {"real": 16000, "morph": 14000},   # 30,000 صورة تدريب
    "val":   {"real": 7000,  "morph": 6000},    # 13,000 صورة تحقق
    "test":  {"real": 1000,  "morph": 1000},    # 2,000  صورة مراقبة/اختبار
}

IMG_SIZE = (256, 256)          # حجم موحّد لجميع الصور النهائية
MORPH_ALPHA = 0.5              # نسبة المزج بين الوجهين (50% / 50%)
MIN_SHARPNESS = 20.0           # حد أدنى لحدّة الصورة (تباين لابلاس) للمراجعة التلقائية

OUT_DIR = "/content/DMorphNet_dataset"
for split in SPLITS:
    os.makedirs(os.path.join(OUT_DIR, split, "real"), exist_ok=True)
    os.makedirs(os.path.join(OUT_DIR, split, "morph"), exist_ok=True)

total_real  = sum(v["real"]  for v in SPLITS.values())
total_morph = sum(v["morph"] for v in SPLITS.values())
print(f"الهدف: {total_real} حقيقية + {total_morph} مدموجة = {total_real + total_morph} صورة")


## ٥) تقسيم الهويات واختيار الصور الحقيقية

**أهم خطوة للعدالة**: نقسّم **الأشخاص (الهويات)** أولاً إلى ثلاث مجموعات منفصلة تماماً، ثم نأخذ الصور من كل مجموعة:

- كل شخص يذهب بكامل صوره إلى قسم واحد فقط (تدريب **أو** تحقق **أو** اختبار).
- صور المورف لاحقاً ستُولَّد فقط من أشخاص **داخل نفس القسم**، فلا تتسرب أي هوية بين الأقسام.


In [ ]:
# تجميع الصور حسب هوية الشخص
by_id = ident.groupby("identity")["image_id"].apply(list).to_dict()
img2id = dict(zip(ident["image_id"], ident["identity"]))

# التأكد من أن الصور موجودة فعلاً على القرص
available = {os.path.basename(p) for p in all_jpgs}
by_id = {k: [f for f in v if f in available] for k, v in by_id.items()}
by_id = {k: v for k, v in by_id.items() if len(v) > 0}

identities = list(by_id.keys())
random.shuffle(identities)

selected = {}
id_iter = iter(identities)
for split, targets in SPLITS.items():
    images, split_ids = [], set()
    while len(images) < targets["real"]:
        pid = next(id_iter)
        split_ids.add(pid)
        images.extend(by_id[pid])
    # نقص الصور الزائدة عن الهدف (الهوية تبقى حصرية لهذا القسم)
    selected[split] = {"identities": split_ids, "images": images[: targets["real"]]}
    print(f"{split}: {len(selected[split]['images'])} صورة حقيقية "
          f"من {len(split_ids)} شخصاً مختلفاً")

# تحقق مبكر: لا تقاطع بين هويات الأقسام
for a in SPLITS:
    for b in SPLITS:
        if a < b:
            inter = selected[a]["identities"] & selected[b]["identities"]
            assert not inter, f"تداخل هويات بين {a} و {b}!"
print("✅ الهويات منفصلة تماماً بين الأقسام الثلاثة")


## ٦) دوال توليد المورف (دمج وجهين في صورة واحدة)

آلية الدمج (نفس مبدأ أدوات AI FaceSwap):
1. استخراج **468 معلماً** لكل وجه باستخدام MediaPipe FaceMesh + 8 نقاط على حواف الصورة.
2. حساب المعالم **الوسطية** بين الوجهين.
3. تقسيم الصورة إلى مثلثات صغيرة (**Delaunay Triangulation**).
4. تشويه (Warp) كل مثلث من الصورتين نحو الشكل الوسطي بتحويل Affine.
5. **مزج** الألوان بين الوجهين بنسبة `alpha = 0.5` — فتنتج صورة واحدة تحمل ملامح الشخصين معاً بشكل واقعي وسلس.


In [ ]:
import cv2
import mediapipe as mp
from scipy.spatial import Delaunay

# واجهة موحدة تعمل مع نسختي MediaPipe:
# القديمة (mp.solutions) والجديدة (Tasks API) التي أُزيلت منها solutions
if hasattr(mp, "solutions"):
    _face_mesh = mp.solutions.face_mesh.FaceMesh(
        static_image_mode=True, max_num_faces=1, min_detection_confidence=0.5)

    def _mesh_points(img_rgb):
        res = _face_mesh.process(img_rgb)
        if not res.multi_face_landmarks:
            return None
        return res.multi_face_landmarks[0].landmark
else:
    import os, urllib.request
    from mediapipe.tasks.python import BaseOptions
    from mediapipe.tasks.python import vision as mp_vision
    _LM_MODEL = "/content/face_landmarker.task"
    if not os.path.exists(_LM_MODEL):
        urllib.request.urlretrieve(
            "https://storage.googleapis.com/mediapipe-models/face_landmarker/"
            "face_landmarker/float16/1/face_landmarker.task", _LM_MODEL)
    _landmarker = mp_vision.FaceLandmarker.create_from_options(
        mp_vision.FaceLandmarkerOptions(
            base_options=BaseOptions(model_asset_path=_LM_MODEL), num_faces=1))

    def _mesh_points(img_rgb):
        mp_img = mp.Image(image_format=mp.ImageFormat.SRGB,
                          data=np.ascontiguousarray(img_rgb))
        res = _landmarker.detect(mp_img)
        return res.face_landmarks[0] if res.face_landmarks else None

print("واجهة MediaPipe المستخدمة:",
      "القديمة (solutions)" if hasattr(mp, "solutions") else "الجديدة (Tasks)")


def get_landmarks(img_bgr):
    # استخراج معالم الوجه + نقاط حواف الصورة. ترجع None إذا لم يوجد وجه.
    lms = _mesh_points(cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB))
    if lms is None:
        return None
    h, w = img_bgr.shape[:2]
    pts = np.array([[lm.x * w, lm.y * h] for lm in lms], dtype=np.float32)
    h1, w1 = h - 1, w - 1
    border = np.array([[0, 0], [w1 // 2, 0], [w1, 0], [w1, h1 // 2],
                       [w1, h1], [w1 // 2, h1], [0, h1], [0, h1 // 2]],
                      dtype=np.float32)
    pts = np.vstack([pts, border])
    pts[:, 0] = np.clip(pts[:, 0], 0, w1)
    pts[:, 1] = np.clip(pts[:, 1], 0, h1)
    return pts


def _warp_triangle(src, src_tri, dst_tri, size):
    M = cv2.getAffineTransform(np.float32(src_tri), np.float32(dst_tri))
    return cv2.warpAffine(src, M, size, None, flags=cv2.INTER_LINEAR,
                          borderMode=cv2.BORDER_REFLECT_101)


def _morph_triangle(img1, img2, out, t1, t2, t, alpha):
    r1 = cv2.boundingRect(np.float32([t1]))
    r2 = cv2.boundingRect(np.float32([t2]))
    r  = cv2.boundingRect(np.float32([t]))
    if r[2] <= 0 or r[3] <= 0 or r1[2] <= 0 or r1[3] <= 0 or r2[2] <= 0 or r2[3] <= 0:
        return
    t1r = [(t1[i][0] - r1[0], t1[i][1] - r1[1]) for i in range(3)]
    t2r = [(t2[i][0] - r2[0], t2[i][1] - r2[1]) for i in range(3)]
    tr  = [(t[i][0]  - r[0],  t[i][1]  - r[1])  for i in range(3)]

    mask = np.zeros((r[3], r[2], 3), dtype=np.float32)
    cv2.fillConvexPoly(mask, np.int32(tr), (1.0, 1.0, 1.0), 16, 0)

    p1 = img1[r1[1]:r1[1] + r1[3], r1[0]:r1[0] + r1[2]]
    p2 = img2[r2[1]:r2[1] + r2[3], r2[0]:r2[0] + r2[2]]
    try:
        w1 = _warp_triangle(p1, t1r, tr, (r[2], r[3]))
        w2 = _warp_triangle(p2, t2r, tr, (r[2], r[3]))
    except cv2.error:
        return
    blended = (1.0 - alpha) * w1 + alpha * w2
    roi = out[r[1]:r[1] + r[3], r[0]:r[0] + r[2]]
    out[r[1]:r[1] + r[3], r[0]:r[0] + r[2]] = roi * (1 - mask) + blended * mask


def morph_faces(img1, img2, alpha=MORPH_ALPHA):
    # دمج وجهين في صورة واحدة. ترجع None عند فشل اكتشاف الوجه.
    img1 = cv2.resize(img1, IMG_SIZE)
    img2 = cv2.resize(img2, IMG_SIZE)
    p1, p2 = get_landmarks(img1), get_landmarks(img2)
    if p1 is None or p2 is None:
        return None
    avg = (1 - alpha) * p1 + alpha * p2
    try:
        triangles = Delaunay(avg).simplices
    except Exception:
        return None
    out = np.zeros(img1.shape, dtype=np.float32)
    f1, f2 = img1.astype(np.float32), img2.astype(np.float32)
    for s in triangles:
        _morph_triangle(f1, f2, out, p1[s], p2[s], avg[s], alpha)
    return np.clip(out, 0, 255).astype(np.uint8)


def sharpness(img_bgr):
    # قياس حدّة الصورة (تباين مرشح لابلاس) — للمراجعة التلقائية.
    return cv2.Laplacian(cv2.cvtColor(img_bgr, cv2.COLOR_BGR2GRAY),
                         cv2.CV_64F).var()

print("✅ دوال الدمج جاهزة")


## ٧) توليد 21,000 صورة مورف مع المراجعة التلقائية

لكل قسم نولّد العدد المطلوب من صور المورف:
- نختار عشوائياً صورتين **لشخصين مختلفين من نفس القسم** (حفاظاً على فصل الهويات).
- ندمجهما، ثم **نراجع** النتيجة تلقائياً ونرفض أي صورة:
  - لا يُكتشف فيها وجه واضح (تبدو مصطنعة/مشوّهة)، أو
  - حدّتها أقل من الحد الأدنى (ضبابية).
- نسجّل لكل صورة مورف: هويتي المصدرين وأسماء الصورتين الأصليتين.

> ⏳ هذه أطول خلية في الدفتر (١–٣ ساعات للأعداد الكاملة). شريط التقدم يوضح الحالة أولاً بأول.


In [ ]:
from tqdm.auto import tqdm

records = []          # سجل الملصقات لكل الصور (حقيقية ومدموجة)
rejected = 0          # عدد الصور المرفوضة في المراجعة التلقائية

for split, targets in SPLITS.items():
    pool = selected[split]["images"]
    morph_dir = os.path.join(OUT_DIR, split, "morph")
    done = 0
    pbar = tqdm(total=targets["morph"], desc=f"توليد مورف {split}")
    while done < targets["morph"]:
        a, b = random.sample(pool, 2)
        if img2id[a] == img2id[b]:      # يجب أن يكونا شخصين مختلفين
            continue
        i1 = cv2.imread(os.path.join(IMG_DIR, a))
        i2 = cv2.imread(os.path.join(IMG_DIR, b))
        if i1 is None or i2 is None:
            continue
        m = morph_faces(i1, i2)
        # --- المراجعة التلقائية (تصفية الصور غير الواقعية) ---
        if m is None or get_landmarks(m) is None or sharpness(m) < MIN_SHARPNESS:
            rejected += 1
            continue
        fname = f"morph_{split}_{done:05d}.jpg"
        cv2.imwrite(os.path.join(morph_dir, fname), m,
                    [cv2.IMWRITE_JPEG_QUALITY, 95])
        records.append({"filename": fname, "split": split, "label": "morph",
                        "src1": a, "src2": b,
                        "id1": img2id[a], "id2": img2id[b]})
        done += 1
        pbar.update(1)
    pbar.close()

print(f"✅ تم توليد {sum(1 for r in records if r['label'] == 'morph')} صورة مورف")
print(f"عدد الصور المرفوضة أثناء المراجعة التلقائية: {rejected}")


## ٨) نسخ الصور الحقيقية إلى المجموعة النهائية

ننسخ الـ 24,000 صورة حقيقية المختارة إلى مجلدات الأقسام بعد توحيد الحجم (256×256)، ونضيف ملصقاتها إلى السجل.


In [ ]:
for split, targets in SPLITS.items():
    real_dir = os.path.join(OUT_DIR, split, "real")
    for fname in tqdm(selected[split]["images"], desc=f"نسخ الصور الحقيقية {split}"):
        img = cv2.imread(os.path.join(IMG_DIR, fname))
        if img is None:
            continue
        out_name = f"real_{fname}"
        cv2.imwrite(os.path.join(real_dir, out_name),
                    cv2.resize(img, IMG_SIZE),
                    [cv2.IMWRITE_JPEG_QUALITY, 95])
        records.append({"filename": out_name, "split": split, "label": "real",
                        "src1": fname, "src2": None,
                        "id1": img2id[fname], "id2": None})

print("✅ تم نسخ جميع الصور الحقيقية")


## ٩) حفظ الملصقات (Labels) والمراجعة البصرية

- نحفظ ملف `labels.csv` الذي يحتوي على: اسم الصورة، القسم، الملصق (حقيقي/مدموج)، وهويات المصدر.
- ثم نعرض عيّنة عشوائية من الصور الحقيقية والمدموجة **للمراجعة اليدوية** والتأكد أن صور المورف تبدو واقعية.


In [ ]:
import matplotlib.pyplot as plt

labels = pd.DataFrame(records)
labels_path = os.path.join(OUT_DIR, "labels.csv")
labels.to_csv(labels_path, index=False)
print("تم حفظ الملصقات في:", labels_path)
print(labels.groupby(["split", "label"]).size())

# عرض عيّنة: صف صور حقيقية وصف صور مدموجة
fig, axes = plt.subplots(2, 6, figsize=(18, 6))
for col, (lab, title) in enumerate([("real", "حقيقية"), ("morph", "مدموجة")]):
    sample = labels[labels.label == lab].sample(6, random_state=SEED)
    for i, (_, row) in enumerate(sample.iterrows()):
        p = os.path.join(OUT_DIR, row.split, row.label, row.filename)
        img = cv2.cvtColor(cv2.imread(p), cv2.COLOR_BGR2RGB)
        axes[col, i].imshow(img)
        axes[col, i].set_title(f"{title} ({row.split})", fontsize=9)
        axes[col, i].axis("off")
plt.tight_layout()
plt.show()


## ١٠) التحقق النهائي من الأعداد وفصل الهويات

نتحقق برمجياً من أن:
1. الأعداد مطابقة تماماً للورقة (45,000 = 24,000 حقيقية + 21,000 مدموجة، وتوزيع الأقسام صحيح).
2. **لا توجد أي هوية مشتركة** بين أي قسمين — سواء من الصور الحقيقية أو من مصادر صور المورف.


In [ ]:
# 1) التحقق من الأعداد
for split, targets in SPLITS.items():
    n_real  = len(labels[(labels.split == split) & (labels.label == "real")])
    n_morph = len(labels[(labels.split == split) & (labels.label == "morph")])
    print(f"{split}: حقيقية={n_real}  مدموجة={n_morph}  الإجمالي={n_real + n_morph}")
    assert n_real == targets["real"] and n_morph == targets["morph"], f"خطأ في أعداد {split}!"

print(f"\nالإجمالي الكلي: {len(labels)} صورة")
assert len(labels) == 45000 or len(labels) == sum(
    t["real"] + t["morph"] for t in SPLITS.values())

# 2) التحقق من فصل الهويات بين الأقسام
def identities_of(split):
    sub = labels[labels.split == split]
    ids = set(sub.id1.dropna().astype(int)) | set(sub.id2.dropna().astype(int))
    return ids

names = list(SPLITS.keys())
for i in range(len(names)):
    for j in range(i + 1, len(names)):
        inter = identities_of(names[i]) & identities_of(names[j])
        assert not inter, f"❌ هويات مشتركة بين {names[i]} و {names[j]}: {len(inter)}"
        print(f"✅ لا توجد هويات مشتركة بين {names[i]} و {names[j]}")

print("\n🎉 مجموعة البيانات جاهزة ومطابقة للمواصفات، والتقييم عادل وواقعي")


## ١١) (اختياري) ضغط المجموعة وحفظها في Google Drive

بما أن جلسة Colab مؤقتة، من الأفضل ضغط المجموعة النهائية ونسخها إلى Google Drive حتى لا تفقدها، ولاستخدامها لاحقاً في تدريب نموذج DMorphNet.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

archive = shutil.make_archive("/content/DMorphNet_dataset", "zip", OUT_DIR)
print("تم إنشاء الأرشيف:", archive)

dest = "/content/drive/MyDrive/DMorphNet_dataset.zip"
shutil.copy(archive, dest)
print("تم الحفظ في Google Drive:", dest)
